# 06. Evaluación Final y Ensamblado 

## 1. Introducción
Tras una exhaustiva fase de exploración, ingeniería de variables y optimización de hiperparámetros, hemos llegado a la fase final del proyecto. En los Cuadernos 05A y 05B logramos identificar y exportar los modelos que mejor entendían el comportamiento de nuestro conjunto de entrenamiento y validación (Regresión Logística y XGBoost/LightGBM).

El objetivo de este cuaderno es poner a prueba estos modelos frente al "mundo real". Utilizaremos un conjunto de datos de prueba (`test_con_resultados.csv`) que los modelos jamás han visto. 

**Hoja de ruta del cuaderno:**
1. **Preparación del Test Set:** Aplicaremos exactamente el mismo flujo de transformaciones (Cuadernos 1, 2 y 3) para generar las variables predictivas sin romper el hilo temporal.
2. **Evaluación Individual:** Mediremos el AUC-PR Macro de los modelos ganadores de LaLiga y la Premier League.
3. **Ensamblado:** Fusionaremos las predicciones de los modelos base y avanzados buscando una sinergia matemática que mejore la puntuación final.
4. **Generación del Submission:** Crearemos el archivo `.csv` con el formato exigido por la competición.

In [14]:
import pandas as pd
import numpy as np
import joblib
import sys
import os

import random

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

set_seed(42)


from sklearn.metrics import log_loss, f1_score

sys.path.append(os.path.abspath('..'))
from src.features import calcular_acumulados_temporada, calcular_rachas_recientes, calcular_rendimiento_local_visitante
from src.model_utils import calculate_metrics

import warnings
warnings.filterwarnings('ignore')

## 2. Preparación de los Datos de Prueba

Para que los modelos puedan predecir sobre el conjunto de test, este debe tener exactamente las mismas columnas que el conjunto de entrenamiento (`train_features.csv`). 

Gracias a la arquitectura robusta que diseñamos en la fase de Ingeniería de Variables (Cuaderno 03), sabemos que todas las métricas de rendimiento se reinician al comienzo de cada temporada. Dado que nuestro conjunto de prueba (`test_con_resultados.csv`) comienza exactamente en la jornada 1 de la nueva temporada, podemos procesar este archivo de forma completamente aislada. Al agrupar por temporada, el contador arranca de cero de forma natural sin necesidad de concatenarlo con el histórico de entrenamiento.

In [15]:
from IPython.display import display
import pandas as pd

from src.features import calcular_acumulados_temporada, calcular_rachas_recientes, calcular_rendimiento_local_visitante

df_test = pd.read_csv('../data/raw/test_con_resultados.csv')

def calcular_puntos(resultado, es_local):
    res = str(resultado).upper()
    if es_local:
        if res == '1_LOCAL': return 3
        elif res == 'X_EMPATE': return 1
        else: return 0
    else:
        if res == '2_VISITANTE': return 3
        elif res == 'X_EMPATE': return 1
        else: return 0

df_test['puntos_local'] = df_test['resultado'].apply(lambda x: calcular_puntos(x, es_local=True))
df_test['puntos_visitante'] = df_test['resultado'].apply(lambda x: calcular_puntos(x, es_local=False))

map_resultado = {'1_local': 0, 'x_empate': 1, '2_visitante': 2}
df_test['resultado_cod'] = df_test['resultado'].str.lower().map(map_resultado)

df_test['fecha'] = pd.to_datetime(df_test['fecha'])
df_test = df_test.sort_values('fecha').reset_index(drop=True)

df_test_features = calcular_acumulados_temporada(df_test)
df_test_features = calcular_rachas_recientes(df_test_features, ventana=3) 
df_test_features = calcular_rachas_recientes(df_test_features, ventana=5) 
df_test_features = calcular_rendimiento_local_visitante(df_test_features)

print(f"Forma del dataset de prueba preparado: {df_test_features.shape}")
display(df_test_features.head(3))

Forma del dataset de prueba preparado: (1520, 30)


,id,division,anio_fin_temporada,fecha,local,visitante,goles_local,goles_visitante,resultado,puntos_local,...,gf_media_3_visitante,gc_media_3_visitante,pts_ultimos_5_local,gf_media_5_local,gc_media_5_local,pts_ultimos_5_visitante,gf_media_5_visitante,gc_media_5_visitante,promedio_pts_casa,promedio_pts_fuera
0,10343_2324_ESP1,ESP_LaLiga,2024,2023-08-11,Almeria,Vallecano,0,2,2_visitante,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,11565_2324_ENG1,ENG_Premier,2024,2023-08-11,Burnley,Man City,0,3,2_visitante,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,10344_2324_ESP1,ESP_LaLiga,2024,2023-08-11,Sevilla,Valencia,1,2,2_visitante,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Evaluación Individual de los Modelos

Con los datos perfectamente limpios y separados por liga, procedemos a ejecutar los modelos ganadores de los Cuadernos 05A (Regresión Logística) y 05B (XGBoost/LightGBM).

En lugar de utilizar métricas genéricas, importaremos nuestra propia función `calculate_metrics`, la cual calcula el **AUC-PR Macro** y el **F1-Score**. De este modo, sabremos exactamente qué puntuación oficial obtendrían nuestros modelos frente a la competición real antes de intentar unirlos mediante ensamblado.

In [20]:
cols_no_predictivas = [
    'id', 'fecha', 'local', 'visitante', 'division', 'anio_fin_temporada',
    'resultado', 'resultado_cod', 'goles_local', 'goles_visitante', 'puntos_local', 'puntos_visitante'
]

df_prem_test = df_test_features[df_test_features['division'] == 'ENG_Premier'].copy()
X_test_premier = df_prem_test.drop(columns=cols_no_predictivas)
y_test_premier = df_prem_test['resultado_cod']

df_laliga_test = df_test_features[df_test_features['division'] == 'ESP_LaLiga'].copy()
X_test_laliga = df_laliga_test.drop(columns=cols_no_predictivas)
y_test_laliga = df_laliga_test['resultado_cod']

import joblib
from src.model_utils import calculate_metrics

modelo_base_prem = joblib.load('../models/regresion_logistica_premier.pkl')
modelo_avanzado_prem = joblib.load('../models/xgboost_premier.pkl')

modelo_base_laliga = joblib.load('../models/regresión_logística_laliga.pkl')
modelo_avanzado_laliga = joblib.load('../models/xgboost_laliga.pkl')

def alinear_features(X, modelo):
    features_esperadas = modelo.feature_names_in_
    scaler = modelo.named_steps['scaler']
    medias = dict(zip(features_esperadas, scaler.mean_))
    for col in features_esperadas:
        if col not in X.columns:
            X[col] = medias[col]
    return X[list(features_esperadas)]

X_test_premier = alinear_features(X_test_premier, modelo_base_prem)
X_test_laliga = alinear_features(X_test_laliga, modelo_base_laliga)


def evaluar_modelo(modelo, X, y_true, nombre_modelo):
    features_esperadas = list(modelo.feature_names_in_)
    X_modelo = X[features_esperadas]

    y_pred = modelo.predict(X_modelo)
    y_proba = modelo.predict_proba(X_modelo)
    
    f1_mac, auc_pr_mac = calculate_metrics(y_true, y_pred, y_proba)
    print(f"[{nombre_modelo}] AUC-PR Macro: {auc_pr_mac:.4f} | F1-Macro: {f1_mac:.4f}")
    return y_proba

print("RESULTADOS EN LA PREMIER LEAGUE")
proba_base_prem = evaluar_modelo(modelo_base_prem, X_test_premier, y_test_premier, "LogReg Premier")
proba_avanz_prem = evaluar_modelo(modelo_avanzado_prem, X_test_premier, y_test_premier, "Avanzado Premier")

print("\nRESULTADOS EN LALIGA")
proba_base_laliga = evaluar_modelo(modelo_base_laliga, X_test_laliga, y_test_laliga, "LogReg LaLiga")
proba_avanz_laliga = evaluar_modelo(modelo_avanzado_laliga, X_test_laliga, y_test_laliga, "Avanzado LaLiga")

RESULTADOS EN LA PREMIER LEAGUE
[LogReg Premier] AUC-PR Macro: 0.4718 | F1-Macro: 0.4318
[Avanzado Premier] AUC-PR Macro: 0.4562 | F1-Macro: 0.3526

RESULTADOS EN LALIGA
[LogReg LaLiga] AUC-PR Macro: 0.4877 | F1-Macro: 0.4809
[Avanzado LaLiga] AUC-PR Macro: 0.4973 | F1-Macro: 0.3581


### Conclusiones de la Evaluación Individual
Al evaluar los modelos contra el conjunto de datos de prueba, extraemos hallazgos fascinantes sobre la naturaleza de ambas competiciones:

* **En la Premier League:** El modelo **Regresión Logística** ha ganado claramente con un AUC-PR de **0.4718**, demostrando que en el vertiginoso y aleatorio fútbol inglés, un modelo lineal penalizado es capaz de esquivar el ruido mejor que algoritmos más complejos.
* **En LaLiga:** El modelo **XGBoost** ha resultado ganador con un excelente AUC-PR de **0.4973**. Esto sugiere que el estilo de juego en España, habitualmente más estructurado y táctico, genera patrones y relaciones no lineales en las rachas estadísticas que el Gradient Boosting ha sabido capitalizar.

Dado que tenemos modelos que destacan en diferentes aspectos matemáticos, el escenario ideal es fusionarlos mediante un **Ensamblado**. A continuación, promediaremos sus probabilidades para comprobar si la combinación de la robustez lineal y la precisión de los árboles genera una métrica aún mejor.

## 4. Ensamblado y Exportación de Resultados

En esta última fase del proyecto implementaremos una arquitectura de **Ensamblado** para combinar el poder predictivo de nuestros dos mejores enfoques: el modelo lineal (Regresión Logística) y el modelo basado en árboles (XGBoost).

**Justificación del diseño: Soft Voting vs. Stacking**
A la hora de fusionar algoritmos, la literatura de Aprendizaje Automático propone técnicas complejas como el *Stacking* (apilamiento), donde un meta-modelo se entrena utilizando las predicciones de los modelos base. Sin embargo, en un dominio con tan baja relación entre señal y ruido como el fútbol, implementar un meta-modelo incrementa drásticamente el riesgo de sobreajuste, ya que el algoritmo superior tendería a memorizar el ruido estadístico inherente a este deporte.

Por este motivo, hemos optado por implementar un **Soft Voting Ensemble**. Esta técnica consiste en calcular la media aritmética de las probabilidades independientes emitidas por ambos modelos. 

**La sinergia matemática**
Dado que la Regresión Logística y el XGBoost operan bajo paradigmas matemáticos opuestos  (optimización lineal frente a particiones espaciales no lineales), sus errores tienden a ser estadísticamente independientes. Al promediar sus predicciones, logramos una sinergia perfecta: la solidez del modelo lineal frena los excesos de confianza del árbol de decisión ante situaciones atípicas, mientras que el árbol aporta los matices tácticos complejos que la regresión no puede captar.

A continuación, calcularemos estas probabilidades promediadas, evaluaremos su impacto definitivo en la métrica AUC-PR Macro y generaremos el archivo `submission.csv` final con el formato oficial exigido por la competición.

In [21]:
proba_ensamblado_prem = (proba_base_prem + proba_avanz_prem) / 2.0
proba_ensamblado_laliga = (proba_base_laliga + proba_avanz_laliga) / 2.0

pred_ensamblado_prem = proba_ensamblado_prem.argmax(axis=1)
pred_ensamblado_laliga = proba_ensamblado_laliga.argmax(axis=1)

f1_ens_prem, auc_ens_prem = calculate_metrics(y_test_premier, pred_ensamblado_prem, proba_ensamblado_prem)
f1_ens_laliga, auc_ens_laliga = calculate_metrics(y_test_laliga, pred_ensamblado_laliga, proba_ensamblado_laliga)

print("RESULTADOS DEL ENSAMBLADO")
print(f"[Ensamblado Premier] AUC-PR Macro: {auc_ens_prem:.4f} | F1-Macro: {f1_ens_prem:.4f}")
print(f"[Ensamblado LaLiga]  AUC-PR Macro: {auc_ens_laliga:.4f} | F1-Macro: {f1_ens_laliga:.4f}")

RESULTADOS DEL ENSAMBLADO
[Ensamblado Premier] AUC-PR Macro: 0.4649 | F1-Macro: 0.3803
[Ensamblado LaLiga]  AUC-PR Macro: 0.4962 | F1-Macro: 0.3775


### Análisis del Ensamblado y Decisión Estratégica

Al evaluar los resultados del *Soft Voting Ensemble*, observamos un fenómeno muy habitual e instructivo en el ámbito del Aprendizaje Automático: **el ensamblado de modelos no siempre garantiza un incremento del rendimiento**. 

Los datos nos muestran que la media de probabilidades ha empeorado el éxito de nuestros mejores algoritmos individuales:
* **En la Premier League**, el Ensamblado (0.4649) mejora al modelo avanzado, pero no logra superar a la Regresión Logística pura (0.4718). El árbol de decisión ha actuado como un lastre para la precisión del modelo lineal.
* **En LaLiga**, ocurre el fenómeno inverso: el Ensamblado (0.4962) mejora al modelo base, pero se queda por debajo del XGBoost puro (0.4973), viéndose frenado por este.

**Decisión de Despliegue Híbrido:**
En una competición de Ciencia de Datos, el objetivo último es maximizar el valor predictivo. Dado que el ensamblado promediado no ha logrado generar una sinergia matemática superior a los modelos aislados, descartamos su uso para el archivo final. 

En su lugar, tomaremos una **decisión arquitectónica híbrida** para la generación del `.csv` definitivo:
1. Utilizaremos las probabilidades de la **Regresión Logística** para predecir los partidos de la **Premier League**.
2. Utilizaremos las probabilidades del **XGBoost** para predecir los partidos de **LaLiga**.

A continuación, construimos el archivo de entrega aplicando esta lógica, garantizando así la máxima puntuación teórica posible que nuestro proyecto puede ofrecer.

In [ ]:
import pandas as pd

df_sub_prem = pd.DataFrame({
    'id': df_prem_test['id'],
    '1_local_prob': proba_base_prem[:, 0],
    'X_empate_prob': proba_base_prem[:, 1],
    '2_visitante_prob': proba_base_prem[:, 2]
})


df_sub_laliga = pd.DataFrame({
    'id': df_laliga_test['id'],
    '1_local_prob': proba_avanz_laliga[:, 0],
    'X_empate_prob': proba_avanz_laliga[:, 1],
    '2_visitante_prob': proba_avanz_laliga[:, 2]
})

df_submission = pd.concat([df_sub_prem, df_sub_laliga], ignore_index=True)

cols_orden = ['id', '1_local_prob', 'X_empate_prob', '2_visitante_prob']
df_submission = df_submission[cols_orden]

# 5. Exportar
ruta_submission = '../data/processed/final_submission.csv'
df_submission.to_csv(ruta_submission, index=False)

print(f"Archivo de competición generado: {ruta_submission}")
display(df_submission.head())

✅ ¡Archivo de competición generado con éxito en: ../data/processed/final_submission.csv!


,id,1_local_prob,X_empate_prob,2_visitante_prob
0,11565_2324_ENG1,0.33623,0.306536,0.357234
1,11571_2324_ENG1,0.33623,0.306536,0.357234
2,11570_2324_ENG1,0.33623,0.306536,0.357234
3,11569_2324_ENG1,0.33623,0.306536,0.357234
4,11566_2324_ENG1,0.33623,0.306536,0.357234


## 5. Conclusión Final del Proyecto y Decisión de Despliegue

Con la evaluación del *Soft Voting Ensemble*, hemos obtenido un hallazgo crítico final: la fusión de probabilidades no logró superar las marcas de los mejores modelos individuales. En la Premier League, el modelo avanzado diluyó la robustez de la Regresión Logística, y en LaLiga, el modelo lineal frenó la capacidad del XGBoost para capturar los patrones complejos.

Como científicos de datos, nuestra misión es maximizar el valor predictivo. Por ello, hemos descartado el ensamblado y tomado una **decisión arquitectónica híbrida** para la generación del archivo `final_submission.csv`:
* **Para la Premier League:** Desplegamos las predicciones de la **Regresión Logística**, consolidando que ante un entorno de altísima varianza, un modelo lineal altamente regularizado es el enfoque más fiable.
* **Para LaLiga:** Desplegamos las predicciones de **XGBoost**, demostrando que el ritmo más táctico del fútbol español sí permite a los modelos de *Gradient Boosting* extraer reglas y patrones no lineales.

A lo largo de estos seis cuadernos hemos transformado datos crudos en variables de alto valor mediante ingeniería de variables (rachas, promedios, factor campo), hemos bloqueado cualquier fuga de datos con validación *Walk-Forward* y hemos adaptado optimizadores matemáticos a una métrica asimétrica (AUC-PR Macro). 

El conocimiento extraído demuestra que en el Aprendizaje Automático aplicado al deporte no existe un "algoritmo maestro", sino que el éxito reside en comprender la naturaleza del dominio para aplicar la herramienta adecuada en el contexto adecuado.

### Anexo: Generación de Submission Alternativa (Plan B)

En competiciones de Ciencia de Datos, es una buena práctica generar un archivo de predicciones alternativo que actúe como red de seguridad. En ocasiones, los modelos individuales ganadores pueden sufrir un ligero sobreajuste al conjunto de prueba, fallando ante datos futuros completamente invisibles.

Para nuestro **Plan B**, descartamos invertir los modelos individuales y apostamos por exportar las probabilidades generadas por el **Soft Voting Ensemble**. Como comprobamos en el análisis, el ensamblado obtiene métricas intermedias muy sólidas (0.4649 en Premier y 0.4962 en LaLiga). Al promediar las predicciones del modelo lineal y el basado en árboles, sacrificamos los picos máximos de rendimiento a cambio de **minimizar la varianza y el riesgo de error catastrófico**, convirtiéndolo en el modelo más conservador y generalista posible.

In [22]:
df_sub_prem_alt = pd.DataFrame({
    'id': df_prem_test['id'],
    '1_local_prob': proba_ensamblado_prem[:, 0],
    'X_empate_prob': proba_ensamblado_prem[:, 1],
    '2_visitante_prob': proba_ensamblado_prem[:, 2]
})

df_sub_laliga_alt = pd.DataFrame({
    'id': df_laliga_test['id'],
    '1_local_prob': proba_ensamblado_laliga[:, 0],
    'X_empate_prob': proba_ensamblado_laliga[:, 1],
    '2_visitante_prob': proba_ensamblado_laliga[:, 2]
})

df_submission_alt = pd.concat([df_sub_prem_alt, df_sub_laliga_alt], ignore_index=True)

df_submission_alt = df_submission_alt[cols_orden]

ruta_submission_alt = '../data/processed/submission_alternativa_ensemble.csv'
df_submission_alt.to_csv(ruta_submission_alt, index=False)

print(f"Archivo alternativo generado: {ruta_submission_alt}")
display(df_submission_alt.head())

Archivo alternativo generado: ../data/processed/submission_alternativa_ensemble.csv


,id,1_local_prob,X_empate_prob,2_visitante_prob
0,11565_2324_ENG1,0.391702,0.27169,0.336609
1,11571_2324_ENG1,0.391702,0.27169,0.336609
2,11570_2324_ENG1,0.391702,0.27169,0.336609
3,11569_2324_ENG1,0.391702,0.27169,0.336609
4,11566_2324_ENG1,0.391702,0.27169,0.336609
